In [1]:
import os
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from torch.utils.data import Dataset, DataLoader
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
import kagglehub


dataset_path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
print("Dataset folder:", dataset_path)


csv_files = glob.glob(os.path.join(dataset_path, "**", "Reviews.csv"), recursive=True)
if not csv_files:
  csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)

if not csv_files:
  raise FileNotFoundError("CSV file not found in the downloaded Kaggle dataset.")

csv_path = csv_files[0]
print("Using file:", csv_path)

df = pd.read_csv(csv_path)
print(df.head())
print("Original shape:", df.shape)

Using Colab cache for faster access to the 'amazon-fine-food-reviews' dataset.
Dataset folder: /kaggle/input/amazon-fine-food-reviews
Using file: /kaggle/input/amazon-fine-food-reviews/Reviews.csv
   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0         

In [3]:
df = df[["Text", "Score"]].dropna()
df = df.rename(columns={"Text": "review"})

# Convert star rating into sentiment labels
def get_sentiment(score):
  if score <= 2:
    return 0  # Negative
  elif score == 3:
    return 1  # Neutral
  else:
    return 2  # Positive

df["sentiment"] = df["Score"].apply(get_sentiment)
df = df[["review", "sentiment"]]

                                    # Keep a manageable, balanced sample so both models can run in a student lab
SAMPLE_SIZE = 12000
if len(df) > SAMPLE_SIZE:
  df = (df.groupby("sentiment", group_keys=False)
          .apply(lambda x: x.sample(
                n=min(len(x), max(1, int(SAMPLE_SIZE * len(x) / len(df)))),
                random_state=SEED
           )))
                                                                                                    # If rounding leaves fewer rows, fill from remaining rows
  if len(df) < SAMPLE_SIZE:
    remaining = (pd.read_csv(csv_path, usecols=["Text", "Score"])
                   .dropna().rename(columns={"Text": "review"}))
    remaining["sentiment"] = remaining["Score"].apply(get_sentiment)
    remaining = remaining[~remaining.index.isin(df.index)]
    extra_n = min(SAMPLE_SIZE - len(df), len(remaining))
    if extra_n > 0:
      df = pd.concat([df, remaining.sample(extra_n, random_state=SEED)])

df = df.drop_duplicates(subset="review").reset_index(drop=True)
df["review"] = df["review"].astype(str).str.lower().str.strip()
df = df[df["review"] != ""].reset_index(drop=True)

print(df.head())
print("\nClass distribution:")
print(df["sentiment"].value_counts().sort_index())

/tmp/ipykernel_2969/4289261716.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


                                              review  sentiment  Score
0  i have an absolute passion for deep, dark hot ...          0    NaN
1  this drink is so "super energy" it's almost fr...          0    NaN
2  i'm sticking with what used to be carnation, n...          0    NaN
3  aspertame causes alot of problems including pr...          0    NaN
4  i ordered these because my local pet store sto...          0    NaN

Class distribution:
sentiment
0    1692
1     877
2    9156
Name: count, dtype: int64


In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["sentiment"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 9380
Testing rows: 2345


In [5]:
# Build vocabulary using ONLY training reviews
all_text = " ".join(train_df["review"].tolist()).split()
word_counts = Counter(all_text)

MAX_VOCAB_SIZE = 20000
vocab = ["<PAD>", "<UNK>"] + [
    word for word, count in word_counts.most_common(MAX_VOCAB_SIZE)
]
word_to_ix = {word: i for i, word in enumerate(vocab)}

MAX_LEN = 100

def encode_text(text, max_len=MAX_LEN):
  tokens = str(text).lower().split()
  encoded = [word_to_ix.get(word, 1) for word in tokens[:max_len]]
  encoded += [0] * (max_len - len(encoded))
  return encoded

class SentimentDataset(Dataset):
  def __init__(self, dataframe):
    self.texts = [encode_text(t) for t in dataframe["review"]]
    self.labels = dataframe["sentiment"].astype(int).tolist()

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return (
        torch.tensor(self.texts[idx], dtype=torch.long),
        torch.tensor(self.labels[idx], dtype=torch.long)
    )

train_dataset = SentimentDataset(train_df)
test_dataset = SentimentDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Vocabulary size:", len(vocab))

Vocabulary size: 20002


In [9]:
class LSTMClassifier(nn.Module):
  def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, output_dim=3):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
    self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, output_dim)
  def forward(self, x):
    embedded = self.embedding(x)
    _, (hidden, _) = self.lstm(embedded)
    return self.fc(hidden[-1])

model_lstm = LSTMClassifier(len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

EPOCHS = 3
for epoch in range(EPOCHS):
  model_lstm.train()
  total_loss = 0

  for texts, labels in train_loader:
    texts, labels = texts.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model_lstm(texts)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {total_loss / len(train_loader):.4f}")

Epoch 3/3, Loss: 0.6508


In [10]:
# Evaluate LSTM on test data
model_lstm.eval()
lstm_predictions = []
lstm_actual = []

with torch.no_grad():
  for texts, labels in test_loader:
    texts = texts.to(device)
    outputs = model_lstm(texts)
    predictions = torch.argmax(outputs, dim=1).cpu().numpy()

    lstm_predictions.extend(predictions)
    lstm_actual.extend(labels.numpy())

print("LSTM Accuracy:", accuracy_score(lstm_actual, lstm_predictions))
print(classification_report(
    lstm_actual, lstm_predictions,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

LSTM Accuracy: 0.7812366737739872
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       338
     Neutral       1.00      0.01      0.01       176
    Positive       0.78      1.00      0.88      1831

    accuracy                           0.78      2345
   macro avg       0.59      0.34      0.30      2345
weighted avg       0.68      0.78      0.69      2345



In [14]:
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_hf = HFDataset.from_pandas(train_df, preserve_index=False)
test_hf = HFDataset.from_pandas(test_df, preserve_index=False)

def tokenize_function(examples):
  return tokenizer(
      examples["review"],
      truncation=True,
      padding="max_length",
      max_length=128
  )

train_hf = train_hf.map(tokenize_function, batched=True)
test_hf = test_hf.map(tokenize_function, batched=True)

train_hf = train_hf.rename_column("sentiment", "labels")
test_hf = test_hf.rename_column("sentiment", "labels")


columns = ["input_ids", "attention_mask", "labels"]
train_hf.set_format(type="torch", columns=columns)
test_hf.set_format(type="torch", columns=columns)

print(train_hf)

Map:   0%|          | 0/9380 [00:00<?, ? examples/s]

Map:   0%|          | 0/2345 [00:00<?, ? examples/s]

Dataset({
    features: ['review', 'labels', 'Score', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 9380
})


In [15]:
model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "Negative", 1: "Neutral", 2: "Positive"},
    label2id={"Negative": 0, "Neutral": 1, "Positive": 2}
)

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=1)
  return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Evaluate BERT
bert_results = trainer.evaluate()
print("BERT test results:", bert_results)

bert_output = trainer.predict(test_hf)
bert_predictions = np.argmax(bert_output.predictions, axis=1)
bert_actual = bert_output.label_ids

print(classification_report(
    bert_actual, bert_predictions,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))